# 01 - Scraping & Tokenization
Scrape reviews from Google Play Store (M-Tix / Cinema 21), then perform:
- Tokenization
- Lowercasing
- Punctuation removal

**App ID:** `lds.cinema21`

In [4]:
# Install dependencies
%pip install google-play-scraper PySastrawi

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import re
import time
import nltk
from google_play_scraper import reviews_all, Sort
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\adhiraga\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\adhiraga\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## 1. Scraping

In [6]:
def scrape_reviews(app_id, langs, country='id'):
    all_dfs = []
    for lang in langs:
        print(f'Scraping lang={lang}, country={country}...')
        start = time.time()
        result = reviews_all(
            app_id,
            lang=lang,
            country=country,
            sort=Sort.NEWEST,
            sleep_milliseconds=0
        )
        elapsed = time.time() - start
        print(f'Done: {len(result)} reviews in {elapsed:.2f}s')
        df_lang = pd.DataFrame(result)
        df_lang['lang'] = lang
        all_dfs.append(df_lang)
    df_all = pd.concat(all_dfs, ignore_index=True)
    print(f'Total: {len(df_all)} reviews')
    return df_all

df_raw = scrape_reviews('lds.cinema21', langs=['id', 'en'])
df_raw = df_raw[['content', 'score']].dropna(subset=['content'])
df_raw = df_raw[df_raw['content'].str.strip() != '']
print(df_raw.shape)
df_raw.head()

Scraping lang=id, country=id...
Done: 14015 reviews in 9.90s
Scraping lang=en, country=id...
Done: 5424 reviews in 2.99s
Total: 19439 reviews
(19438, 2)


,content,score
0,sangat memudahkan,5
1,seru,4
2,oke,4
3,sangat baik,5
4,lebih gampamg pesan,5


## 2. Save Raw Data

In [8]:
df_raw.to_csv('../data/raw/mtix_raw.csv', index=False)
print(f'Raw data saved: {df_raw.shape[0]} rows')

Raw data saved: 19438 rows


## 3. Tokenization

In [9]:
df_raw['tokenized'] = df_raw['content'].apply(word_tokenize)
df_raw[['content', 'tokenized']].head()

,content,tokenized
0,sangat memudahkan,"[sangat, memudahkan]"
1,seru,[seru]
2,oke,[oke]
3,sangat baik,"[sangat, baik]"
4,lebih gampamg pesan,"[lebih, gampamg, pesan]"


## 4. Lowercasing

In [10]:
df_raw['lower_tokens'] = df_raw['tokenized'].apply(
 lambda tokens: [t.lower() for t in tokens]
)
df_raw[['tokenized', 'lower_tokens']].head()

,tokenized,lower_tokens
0,"[sangat, memudahkan]","[sangat, memudahkan]"
1,[seru],[seru]
2,[oke],[oke]
3,"[sangat, baik]","[sangat, baik]"
4,"[lebih, gampamg, pesan]","[lebih, gampamg, pesan]"


## 5. Punctuation Removal

In [11]:
def remove_punctuation(tokens):
    return [t for t in tokens if re.match(r'[a-zA-Z0-9]', t)]

df_raw['clean_tokens'] = df_raw['lower_tokens'].apply(remove_punctuation)
df_raw[['lower_tokens', 'clean_tokens']].head()

,lower_tokens,clean_tokens
0,"[sangat, memudahkan]","[sangat, memudahkan]"
1,[seru],[seru]
2,[oke],[oke]
3,"[sangat, baik]","[sangat, baik]"
4,"[lebih, gampamg, pesan]","[lebih, gampamg, pesan]"


## 6. Save Tokenized Data

In [12]:
df_raw.to_csv('../data/preprocessed/mtix_tokenized.csv', index=False)
print('Tokenized data saved.')
df_raw[['content', 'score', 'clean_tokens']].head(10)

Tokenized data saved.


,content,score,clean_tokens
0,sangat memudahkan,5,"[sangat, memudahkan]"
1,seru,4,[seru]
2,oke,4,[oke]
3,sangat baik,5,"[sangat, baik]"
4,lebih gampamg pesan,5,"[lebih, gampamg, pesan]"
5,alhamdulillah sdh diuinstall. aplikasi gk berg...,1,"[alhamdulillah, sdh, diuinstall, aplikasi, gk,..."
6,bagus,5,[bagus]
7,ok,5,[ok]
8,memudahkan untuk mengatur waktu sebelum menont...,5,"[memudahkan, untuk, mengatur, waktu, sebelum, ..."
9,udah bisa,5,"[udah, bisa]"
